# DSPy — Otimização com `LabeledFewShot`

Neste notebook será demonstrado o uso do otimizador `LabeledFewShot` do DSPy em um problema de **classificação binária de textos**.

Será utilizada a base **Natural Language Processing with Disaster Tweets**, disponibilizada no Kaggle. O objetivo é classificar cada tweet em uma das seguintes categorias:

* `0`: o tweet **não descreve um desastre real**;
* `1`: o tweet **descreve um desastre real**.

O experimento será dividido em duas etapas:

1. avaliar um classificador DSPy sem exemplos de demonstração (**zero-shot**);
2. otimizar o classificador utilizando `LabeledFewShot` e avaliá-lo novamente.

A métrica principal utilizada para comparar os dois programas será o **F1-score**.

In [1]:
import os
from dotenv import load_dotenv  # Carrega variáveis de ambiente do arquivo .env
import dspy  # Framework para otimização de prompts com Language Models

import pandas as pd

from typing import Literal
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

from tqdm.auto import tqdm

/home/leonardo/Documentos/github/dspy_studies/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()  # Lê variáveis do arquivo .env

True

## Setup - Configuração do Modelo

Primeiro, carregamos as variáveis de ambiente (como API key) do arquivo `.env` na raiz do projeto. Isso evita hardcoding de credenciais no código.

In [3]:
lm = dspy.LM(
    "openai/gpt-5-mini",  # Modelo OpenAI GPT-5 Mini (modelo compacto e rápido)
    api_key=os.getenv("OPENAI_API_KEY"),  # API key carregada da variável de ambiente
)

# Configura o modelo padrão para todas as operações DSPy
dspy.configure(lm=lm)

## 3. Leitura da base de dados

Será utilizada a base da competição **Natural Language Processing with Disaster Tweets**, do Kaggle.

Referência:

https://www.kaggle.com/competitions/nlp-getting-started

Para este experimento são relevantes principalmente duas colunas:

| Coluna   | Descrição                   |
| -------- | --------------------------- |
| `text`   | Texto do tweet              |
| `target` | Classe correta (`0` ou `1`) |

O problema consiste, portanto, em aprender a relação:

`text → target`


In [4]:
df = pd.read_csv('disaster_tweets.csv')
df = df.head(200)
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


## Análise da distribuição das classes

Antes da divisão dos dados, é importante verificar quantos exemplos existem de cada classe.

Além da quantidade absoluta, será analisada a proporção entre tweets classificados como `0` e `1`.

Essa análise é importante porque o **F1-score** considera conjuntamente precisão e recall e é especialmente útil quando existe algum grau de desbalanceamento entre as classes.


In [5]:
df["target"].value_counts()

target
0    103
1     97
Name: count, dtype: int64

In [6]:
df["target"].value_counts(normalize=True)

target
0    0.515
1    0.485
Name: proportion, dtype: float64

## Separação entre treino e teste

A base será dividida em:

* **80% para treinamento**;
* **20% para teste**.

O parâmetro `stratify=df["target"]` é utilizado para manter aproximadamente a mesma proporção entre as classes `0` e `1` nos dois conjuntos.

O conjunto de **treino** poderá ser utilizado pelo otimizador para selecionar exemplos de demonstração.

O conjunto de **teste**, por outro lado, será mantido separado e utilizado exclusivamente para medir a qualidade dos programas.

Essa separação evita que exemplos utilizados durante a otimização sejam empregados também na avaliação final.


In [7]:
df_train, df_test = train_test_split(
    df[["text", "target"]],
    test_size=0.20,
    random_state=42,
    stratify=df["target"],
)

print(f"Treino: {len(df_train)} exemplos")
print(f"Teste:  {len(df_test)} exemplos")

Treino: 160 exemplos
Teste:  40 exemplos


## Conversão para `dspy.Example`

O DSPy representa exemplos de treino e teste por meio da classe `dspy.Example`.

Neste problema, cada exemplo possui dois campos:

* `text`: entrada fornecida ao modelo;
* `target`: resposta esperada.

A chamada:

`with_inputs("text")`

informa explicitamente ao DSPy que `text` deve ser utilizado como entrada do programa.

Consequentemente, `target` passa a ser tratado como o **label**, ou seja, a resposta esperada para aquele exemplo.

Conceitualmente, cada registro passa a ter a seguinte estrutura:

`entrada: text → saída esperada: target`

In [8]:
def dataframe_para_dspy(dataframe):
    exemplos = []

    for _, row in dataframe.iterrows():
        exemplo = dspy.Example(
            text=row["text"],
            target=int(row["target"]),
        ).with_inputs("text")

        exemplos.append(exemplo)

    return exemplos

In [9]:
trainset = dataframe_para_dspy(df_train)
testset = dataframe_para_dspy(df_test)

print(f"Trainset DSPy: {len(trainset)}")
print(f"Testset DSPy:  {len(testset)}")

Trainset DSPy: 160
Testset DSPy:  40


In [10]:
trainset[0]

Example({'text': 'Accident center lane blocked in #SantaClara on US-101 NB before Great America Pkwy #BayArea #Traffic http://t.co/pmlOhZuRWR', 'target': 1}) (input_keys={'text'})

## Definição da tarefa com uma `Signature`

No DSPy, uma `Signature` descreve declarativamente a tarefa que será executada pelo modelo.

A `ClassificarTweet` possui:

* um `InputField` chamado `text`, contendo o tweet;
* um `OutputField` chamado `target`, contendo a classificação.

O tipo:

`Literal[0, 1]`

restringe a resposta esperada às duas classes válidas do problema.

Dessa forma, a Signature define claramente o contrato:

`texto do tweet → 0 ou 1`


In [11]:
class ClassificarTweet(dspy.Signature):
    """
    Determine se o tweet descreve um desastre real.

    Retorne:
    - 1 se o tweet estiver relacionado a um desastre real.
    - 0 caso contrário.
    """

    text: str = dspy.InputField(
        desc="Texto do tweet que deve ser classificado."
    )

    target: Literal[0, 1] = dspy.OutputField(
        desc="1 para desastre real e 0 para não desastre."
    )

In [12]:
classificador_base = dspy.Predict(ClassificarTweet)

In [13]:
def avaliar_classificador(programa, dataset, descricao="Avaliando"):
    """
    Executa um programa DSPy sobre um dataset e calcula
    métricas globais de classificação.
    """

    y_true = []
    y_pred = []

    for exemplo in tqdm(dataset, desc=descricao):

        # Executa o programa utilizando somente os campos
        # marcados como entrada pelo with_inputs(...)
        predicao = programa(**exemplo.inputs())

        # Label verdadeiro
        y_true.append(int(exemplo.target))

        # Label previsto pelo DSPy
        y_pred.append(int(predicao.target))

    # Calcula as métricas sobre todo o conjunto
    resultado = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }

    return resultado

In [14]:
resultado_base = avaliar_classificador(
    classificador_base,
    testset,
    descricao="Baseline zero-shot",
)

Baseline zero-shot: 100%|███████████████████████████████████| 40/40 [00:02<00:00, 17.63it/s]


## Avaliação do baseline

O classificador inicial será executado sobre todos os exemplos do conjunto de teste.

Para cada exemplo:

1. o campo `text` é enviado ao programa;
2. o programa produz uma previsão para `target`;
3. a previsão é comparada com o `target` verdadeiro.

Ao final são calculadas as métricas globais de classificação.

Esse resultado será considerado o desempenho **antes da otimização**.


In [15]:
print(f"F1 baseline: {resultado_base['f1']:.4f}")

F1 baseline: 0.9474


In [16]:
print(f"Accuracy:  {resultado_base['accuracy']:.4f}")
print(f"Precision: {resultado_base['precision']:.4f}")
print(f"Recall:    {resultado_base['recall']:.4f}")
print(f"F1:        {resultado_base['f1']:.4f}")

Accuracy:  0.9500
Precision: 0.9474
Recall:    0.9474
F1:        0.9474


## Otimização com `LabeledFewShot`

O `LabeledFewShot` é um dos otimizadores mais simples do DSPy.

Neste experimento será utilizado:

`dspy.LabeledFewShot(k=16)`

O parâmetro `k` determina o número máximo de exemplos rotulados que serão adicionados ao programa como **demonstrações few-shot**.

Quando utilizamos:

`sample=True`

o DSPy seleciona uma amostra dos exemplos existentes no `trainset`.

Portanto, o `LabeledFewShot` **não altera os pesos do modelo de linguagem**. A otimização acontece pela inclusão de exemplos de entrada e saída no contexto utilizado pelo programa.

Conceitualmente:

**Antes**

`instrução + novo tweet → modelo → classificação`

**Depois**

`instrução + exemplos rotulados + novo tweet → modelo → classificação`


In [17]:
optimizer = dspy.LabeledFewShot(
    k=16
)

## Compilação do programa otimizado

O método `compile()` recebe:

* o programa original (`student`);
* o conjunto de treinamento (`trainset`);
* a estratégia de seleção dos exemplos.

O resultado é uma nova versão do programa contendo as demonstrações selecionadas pelo `LabeledFewShot`.

O programa original pode continuar sendo utilizado como baseline, enquanto o programa compilado será utilizado para medir o efeito do few-shot.


In [18]:
classificador_otimizado = optimizer.compile(
    classificador_base,
    trainset=trainset,
    sample=True,
)

## Inspeção das demonstrações selecionadas

Depois da compilação, as demonstrações adicionadas ao programa podem ser acessadas pela propriedade:

`demos`

A inspeção desses exemplos permite visualizar concretamente como o `LabeledFewShot` modificou o programa.

Cada demonstração contém:

* o texto de entrada (`text`);
* a classificação correta (`target`).

Esses exemplos passam a fazer parte do contexto fornecido ao modelo quando uma nova classificação é realizada.


In [19]:
demos = classificador_otimizado.demos

print(f"Número de demonstrações: {len(demos)}")

Número de demonstrações: 16


In [20]:
for i, exemplo in enumerate(demos, start=1):
    print(f"--- Exemplo {i} ---")
    print(f"Tweet:  {exemplo.text}")
    print(f"Target: {exemplo.target}")
    print()

--- Exemplo 1 ---
Tweet:  #GrowingUpSpoiled going clay pigeon shooting and crying because of the 'aftershock'
Target: 0

--- Exemplo 2 ---
Tweet:  Love my girlfriend
Target: 0

--- Exemplo 3 ---
Tweet:  @flowri were you marinading it or was it an accident?
Target: 0

--- Exemplo 4 ---
Tweet:  Tried orange aftershock today. My life will never be the same
Target: 0

--- Exemplo 5 ---
Tweet:  Aashiqui Actress Anu Aggarwal On Her Near-Fatal Accident http://t.co/6Otfp31LqW
Target: 1

--- Exemplo 6 ---
Tweet:  What a goooooooaaaaaal!!!!!!
Target: 0

--- Exemplo 7 ---
Tweet:  I gained 3 followers in the last week. You? Know your stats and grow with http://t.co/TIyUliF5c6
Target: 0

--- Exemplo 8 ---
Tweet:  @20skyhawkmm20 @traplord_29 @FREDOSANTANA300 @LilReese300 it was hella crazy 3 fights an ambulance and a couple mosh pits ??
Target: 1

--- Exemplo 9 ---
Tweet:  #BREAKING: there was a deadly motorcycle car accident that happened to #Hagerstown today. I'll have more details at 5 @Your4Stat

In [21]:
resultado_otimizado = avaliar_classificador(
    classificador_otimizado,
    testset,
    descricao="LabeledFewShot",
)

LabeledFewShot: 100%|███████████████████████████████████████| 40/40 [00:00<00:00, 51.75it/s]


In [22]:
print(f"F1 otimizado: {resultado_otimizado['f1']:.4f}")

F1 otimizado: 0.9474


## Avaliação após a otimização

O programa otimizado será avaliado utilizando **exatamente o mesmo conjunto de teste utilizado pelo baseline**.

Isso permite uma comparação justa entre:

* classificador original;
* classificador com `LabeledFewShot`.

Nenhum exemplo do conjunto de teste participa da seleção das demonstrações.

Ao final, será novamente calculado o F1-score juntamente com Accuracy, Precision e Recall.


In [23]:
comparacao = pd.DataFrame(
    {
        "Modelo": [
            "Baseline (zero-shot)",
            "LabeledFewShot (k=16)",
        ],
        "Accuracy": [
            resultado_base["accuracy"],
            resultado_otimizado["accuracy"],
        ],
        "Precision": [
            resultado_base["precision"],
            resultado_otimizado["precision"],
        ],
        "Recall": [
            resultado_base["recall"],
            resultado_otimizado["recall"],
        ],
        "F1": [
            resultado_base["f1"],
            resultado_otimizado["f1"],
        ],
    }
)

comparacao

,Modelo,Accuracy,Precision,Recall,F1
0,Baseline (zero-shot),0.95,0.947368,0.947368,0.947368
1,LabeledFewShot (k=16),0.95,0.947368,0.947368,0.947368


In [24]:
print("BASELINE")
print(
    classification_report(
        resultado_base["y_true"],
        resultado_base["y_pred"],
        digits=4,
    )
)

BASELINE
              precision    recall  f1-score   support

           0     0.9524    0.9524    0.9524        21
           1     0.9474    0.9474    0.9474        19

    accuracy                         0.9500        40
   macro avg     0.9499    0.9499    0.9499        40
weighted avg     0.9500    0.9500    0.9500        40



In [25]:
print("LABELED FEW SHOT")
print(
    classification_report(
        resultado_otimizado["y_true"],
        resultado_otimizado["y_pred"],
        digits=4,
    )
)

LABELED FEW SHOT
              precision    recall  f1-score   support

           0     0.9524    0.9524    0.9524        21
           1     0.9474    0.9474    0.9474        19

    accuracy                         0.9500        40
   macro avg     0.9499    0.9499    0.9499        40
weighted avg     0.9500    0.9500    0.9500        40



## Persistência do programa otimizado

Após a otimização, o estado do programa será salvo em:

`LabeledFewShot.json`

Ao salvar um módulo DSPy dessa maneira, o arquivo JSON armazena o **estado do programa**, incluindo as demonstrações adicionadas durante a otimização.

Isso permite reutilizar o resultado posteriormente sem precisar executar novamente o `LabeledFewShot`.


In [26]:
classificador_otimizado.save("LabeledFewShot.json")

In [27]:
# Recria a mesma arquitetura do programa
classificador_carregado = dspy.Predict(ClassificarTweet)

# Carrega o estado otimizado salvo pelo LabeledFewShot
classificador_carregado.load("LabeledFewShot.json")
classificador_carregado

Predict(StringSignature(text -> target
    instructions='Determine se o tweet descreve um desastre real.\n\nRetorne:\n- 1 se o tweet estiver relacionado a um desastre real.\n- 0 caso contrário.'
    text = Field(annotation=str required=True json_schema_extra={'desc': 'Texto do tweet que deve ser classificado.', '__dspy_field_type': 'input', 'prefix': 'Text:'})
    target = Field(annotation=Literal[0, 1] required=True json_schema_extra={'desc': '1 para desastre real e 0 para não desastre.', '__dspy_field_type': 'output', 'prefix': 'Target:'})
))

In [28]:
predicao = classificador_carregado(
    text="A massive wildfire is spreading through the forest."
)

print(predicao)

Prediction(
    target=1
)


In [29]:
len(classificador_carregado.demos)

16

In [30]:
# Mostra última chamada ao modelo (n=1 significa 1 última chamada)
dspy.inspect_history(n=1)





[2026-09-07T18:11:10.134079]

System message:

Your input fields are:
1. `text` (str): Texto do tweet que deve ser classificado.
Your output fields are:
1. `target` (Literal[0, 1]): 1 para desastre real e 0 para não desastre.
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## text ## ]]
{text}

Outputs will be a JSON object with the following fields.

{
  "target": "{target}        # note: the value you produce must exactly match (no extra characters) one of: 0; 1"
}
In adhering to this structure, your objective is: 
        Determine se o tweet descreve um desastre real.
        
        Retorne:
        - 1 se o tweet estiver relacionado a um desastre real.
        - 0 caso contrário.


User message:

[[ ## text ## ]]
#GrowingUpSpoiled going clay pigeon shooting and crying because of the 'aftershock'


Assistant message:

{
  "target": 0
}


User message:

[[ ## text ## ]]
Love my girl